# Cuaderno 23: explicación de la lista del objetivo 6

Descriptivo (sección 11 del preregistro; la sección 12 lo declara como desviación). Para los 3.613 procesos de la lista: (a) la parte de cada variable en el error del autoencoder y (b) los valores de Shapley de Isolation Forest.

**Antes de correr:** guarda este cuaderno (Ctrl+S) y no uses "Ejecutar todo". Corre las celdas **en orden, una por una**: E1 a E5 preparan y verifican; **E7 vuelve a ajustar los modelos con la semilla congelada y se detiene si no reproducen la lista**; E8 y E9 explican; E10 escribe las tablas. Si E7 se detiene, no sigas: pega la salida en el chat.

No abre `planA_validacion_sri`, la muestra de H3 ni la clave; no imprime RUC ni procesos. El archivo por proceso (`explicacion_por_proceso.csv`) se queda en tu computadora; al chat solo va `resumen_para_chat.txt`.

In [ ]:
# =====================================================================================
# CUADERNO 23 (EXPLICACION) · CELDA E1. Configuración y verificaciones
#   Qué hace este cuaderno (análisis descriptivo de la sección 11 del preregistro, que la sección 12
#   declara como desviación): para los procesos de la lista del objetivo 6 (columna `priorizado` de la
#   corrida congelada), (a) el error de reconstrucción por variable del autoencoder y (b) los valores
#   de Shapley de Isolation Forest. ANTES vuelve a ajustar los modelos con la semilla de la corrida
#   congelada y comprueba que reproducen sus puntajes; si no los reproducen, no explica nada.
#   Antes de correr: guardar el cuaderno (Ctrl+S); la celda toma la huella de su código desde el archivo.
#   No carga planA_validacion_sri, ni la muestra de H3, ni la clave; no imprime RUC ni procesos.
# =====================================================================================
import os, time, json, math, hashlib, platform, warnings, unicodedata
from datetime import datetime
import numpy as np
import pandas as pd
import duckdb
import sklearn
from scipy.stats import spearmanr, norm
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning

SALIDA_TXT, CELDA_ACTUAL = {}, "E1"
def say(*args):
    texto = " ".join(str(a) for a in args)
    print(texto)
    SALIDA_TXT.setdefault(CELDA_ACTUAL, []).append(texto)

def empezar(nombre):
    global CELDA_ACTUAL
    guardar_txt()
    CELDA_ACTUAL = nombre
    SALIDA_TXT[nombre] = []

def guardar_txt():
    if "DIR_RUN" in globals():
        with open(os.path.join(DIR_RUN, "resumen_para_chat.txt"), "w", encoding="utf-8") as f:
            for celda, lineas in SALIDA_TXT.items():
                f.write(f"\n##### {celda} #####\n" + "\n".join(lineas) + "\n")

# --- rutas ---
SERCOP   = r"C:\SERCOP_FINAL"
BASE     = os.path.join(SERCOP, "plan_A")
SALIDA   = os.path.join(SERCOP, "modelos")
PREREG   = os.path.join(SERCOP, "preregistro")
CUADERNO = os.path.join(BASE, "cuaderno23_explicacion.ipynb")
CONGELADA = os.path.join(SALIDA, "20261016_1017_completo_normal_sup", "puntajes.parquet")
LISTA_CSV = os.path.join(SALIDA, "20261121_2235_objetivo6", "lista_objetivo6.csv")
HUELLA_CONGELADA_ESPERADA = "eed4fbceb2cc9f3c"     # puntajes.parquet de la lista congelada (preregistro_firma.json)
HUELLA_LISTA_ESPERADA     = "ad44ce9a570e0082"     # lista_objetivo6.csv (21-nov-2026)
N_LISTA_ESPERADO = 3613

# --- parámetros del preregistro (iguales al cuaderno 17; no se cambian) ---
MODO = "COMPLETO"
REPRESENTACION = "normal_sup"
SEMILLA = 20261001
IF_ARBOLES, IF_SUBMUESTRA, LOF_VECINOS = 100, 256, 20
CORTE_TOP, MIN_PATRON, ANIO_INICIO = 0.01, 1000, 2016
MIN_GRUPO, REF_MAX, AE_MAX_ENTRENO, HUERFANO_MAX = 100, 50_000, 50_000, 10_000

# --- parámetros de este cuaderno ---
TOL_REPRODUCCION = 1e-9      # diferencia máxima aceptada entre el puntaje reajustado y el congelado
N_CADENAS = 1000             # Shapley: cadenas por proceso (la mitad son permutaciones; cada una con su inversa). Par.
SEMILLA_SHAP = SEMILLA       # semilla de los fondos y las permutaciones del Shapley
BLOQUE_SHAP = 40             # procesos por bloque (solo controla la memoria)
N_FONDO_EXACTO, N_PROC_EXACTO, N_CADENAS_EXACTO = 100, 4, 4000   # control contra el Shapley exacto

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
T_INICIO = time.time()
ALERTAS = []

def sha_completa(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

if MODO != "COMPLETO" or REPRESENTACION != "normal_sup":
    raise SystemExit("Este cuaderno solo corre en modo COMPLETO con normal_sup, como la corrida congelada.")

# preregistro firmado y nada cambiado desde la firma (como en el cuaderno 17)
FIRMA = json.load(open(os.path.join(PREREG, "preregistro_firma.json"), encoding="utf-8"))
if sha_completa(os.path.join(PREREG, FIRMA["documento"])) != FIRMA["huella_documento"]:
    raise SystemExit("El preregistro cambió después de firmarlo: para y avisa.")
for rel, h in FIRMA["huellas_citadas"].items():
    if sha_completa(os.path.join(SERCOP, rel)) != h:
        raise SystemExit(f"{rel} cambió después de la firma: para y avisa.")
# la lista congelada y la lista del objetivo 6 son las esperadas
HUELLA_CONGELADA = sha_completa(CONGELADA)
if not HUELLA_CONGELADA.startswith(HUELLA_CONGELADA_ESPERADA):
    raise SystemExit("puntajes.parquet no es el de la lista congelada: para y avisa.")
HUELLA_LISTA = sha_completa(LISTA_CSV)
if not HUELLA_LISTA.startswith(HUELLA_LISTA_ESPERADA):
    raise SystemExit("lista_objetivo6.csv no es el de la corrida del 21-nov: para y avisa.")

# huella del código de este cuaderno (celdas cuyo título empieza con "# CUADERNO 23 (EXPLICACION)")
def huella_codigo_cuaderno():
    nb = json.load(open(CUADERNO, encoding="utf-8"))
    partes = []
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        src = c["source"] if isinstance(c["source"], str) else "".join(c["source"])
        if any(l.startswith("# CUADERNO 23 (EXPLICACION)") for l in src.splitlines()[:3]):
            partes.append(src)
    return hashlib.sha256("\n# ----- fin de celda -----\n".join(partes).encode("utf-8")).hexdigest(), len(partes)
if not os.path.exists(CUADERNO):
    raise SystemExit(f"No encuentro {CUADERNO}: guarda el cuaderno con ese nombre.")
HUELLA_CODIGO, N_CELDAS = huella_codigo_cuaderno()

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M") + "_explicacion"
DIR_RUN = os.path.join(SALIDA, RUN_ID)
if os.path.exists(DIR_RUN):
    raise SystemExit(f"{DIR_RUN} ya existe (corriste dos veces en el mismo minuto): espera un minuto y vuelve a correr E1.")
os.makedirs(DIR_RUN)

VERSIONES = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                 sklearn=sklearn.__version__, duckdb=duckdb.__version__)
say(f"Corrida {RUN_ID} | preregistro verificado ({FIRMA['huella_documento'][:16]}) | puntajes congelados "
    f"{HUELLA_CONGELADA[:16]} | lista del objetivo 6 {HUELLA_LISTA[:16]}")
say(f"Código de este cuaderno {HUELLA_CODIGO[:16]} ({N_CELDAS} celdas)")
say(" | ".join(f"{k} {v}" for k, v in VERSIONES.items()) + f" | núcleos {os.cpu_count()}")
say(f"Carpeta de salida: {DIR_RUN}")

In [ ]:
# =====================================================================================
# CUADERNO 23 (EXPLICACION) · CELDA E2. Carga de la base y grupos comparables (igual a la celda 2 del cuaderno 17)
#          (NO se carga planA_validacion_sri: la marca del SRI se usa una sola vez, al final)
# =====================================================================================
empezar("E2")
def ruta(nombre):
    return os.path.join(BASE, nombre).replace("\\", "/")

def huella(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()[:n]

def a_float(s):
    """Convierte booleanos, enteros o nulos a float (nulo -> NaN) sin errores de tipo."""
    try:
        return pd.Series(pd.array(s, dtype="Float64").to_numpy(dtype="float64", na_value=np.nan), index=s.index)
    except Exception:
        return pd.to_numeric(s.astype("object"), errors="coerce").astype("float64")

def guardar_parquet(tabla, nombre):
    """Guarda en la carpeta de la corrida; si pandas no tiene motor de parquet, usa DuckDB."""
    destino = os.path.join(DIR_RUN, nombre)
    try:
        tabla.to_parquet(destino)
    except ImportError:
        c = duckdb.connect()
        c.register("t_guardar", tabla.reset_index())
        c.execute(f"COPY t_guardar TO '{destino.replace(chr(92), '/')}' (FORMAT PARQUET)")
        c.close()

def sin_tildes(texto):
    t = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

t_celda = time.time()
IX = ruta("planA_indicadores.parquet")
PM = ruta("planA_procesos_modelo.parquet")
for p in (IX, PM):
    if not os.path.exists(p):
        raise SystemExit(f"No encuentro {p}. Revisa BASE en la celda 1.")
HUELLA_IX, HUELLA_PM = huella(IX), huella(PM)

con = duckdb.connect()
ix = con.execute(f"SELECT * FROM read_parquet('{IX}')").df()
cols_pm = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{PM}')").fetchall()]
pedir = ["ocid"] + [c for c in ["familia", "fecha_inicio_tender", "monto_final", "entidad_ruc", "proveedor_ruc_final"]
                    if c in cols_pm and c not in ix.columns]
pm = con.execute(f"SELECT {', '.join(pedir)} FROM read_parquet('{PM}')").df()
say(f"planA_indicadores: {ix.shape[0]:,} filas x {ix.shape[1]} columnas | huella {HUELLA_IX}")
say(f"planA_procesos_modelo: {pm.shape[0]:,} filas; se toman {pedir[1:]} | huella {HUELLA_PM}")
say("\nColumnas de planA_indicadores:")
say(", ".join(ix.columns))

df = ix.merge(pm, on="ocid", how="left")
if len(df) != len(ix) or df["ocid"].duplicated().any():
    raise SystemExit("La unión por ocid cambió el número de filas: revisar antes de seguir.")
say(f"\nProcesos de indicadores sin pareja en procesos_modelo: {(~ix['ocid'].isin(pm['ocid'])).sum():,}")
if "familia" not in df.columns:
    raise SystemExit("No encuentro la columna 'familia' en ninguno de los dos archivos.")

# fecha del proceso = inicio del período de ofertas
col_fecha = next((c for c in ["fecha_inicio_tender", "fecha_proceso"] if c in df.columns), None)
if col_fecha is None:
    raise SystemExit("No encuentro la fecha del proceso (fecha_inicio_tender).")
fecha = pd.to_datetime(df[col_fecha], errors="coerce")
try:
    if fecha.dt.tz is not None:
        fecha = fecha.dt.tz_convert("America/Guayaquil").dt.tz_localize(None)
except AttributeError:
    pass
say(f"Fecha del proceso tomada de '{col_fecha}'; procesos sin fecha: {int(fecha.isna().sum()):,}")
df["anio_proc"] = fecha.dt.year
df["periodo_calc"] = np.where(fecha >= pd.Timestamp("2025-10-07"), "post", "pre")
if "periodo" in df.columns:
    coinc = (df["periodo"].astype(str).str.lower().str.startswith("post") == (df["periodo_calc"] == "post")).mean()
    say(f"Coincidencia entre 'periodo' de la base y el corte por fecha: {coinc:.4%}")
df["grupo"] = df["familia"].astype(str) + " | " + df["periodo_calc"]
if "monto_final" not in df.columns:
    df["monto_final"] = np.nan
df["monto_final"] = a_float(df["monto_final"])

# Control de calibración sobre la base completa (antes de quitar 2015 y de muestrear):
# cada percentil de la base debe tener media 0,500 dentro de su grupo
say("\nControl de calibración de la base (media de cada percentil dentro de su grupo; debe dar 0,500):")
filas = []
for c in ["pct_precio_alto", "pct_monto_capital", "pct_proveedor_nuevo", "pct_plazo_corto", "pct_firma_rapida",
          "pct_fraccionamiento_pos", "pct_cuota_pos", "pct_recurrencia_pos"]:
    if c in df.columns:
        s_ = a_float(df[c])
        g_ = s_.groupby(df["grupo"])
        m = g_.mean()[g_.count() >= 200]                 # solo grupos con al menos 200 valores
        filas.append(dict(columna=c, media_min=m.min(), media_max=m.max(),
                          grupo_mas_lejos=(m - 0.5).abs().idxmax() if m.notna().any() else "-"))
cal = pd.DataFrame(filas)
say(cal.round(4).to_string(index=False))
lejos = cal[(cal["media_min"] < 0.495) | (cal["media_max"] > 0.505)]
if len(lejos):
    ALERTAS.append("Percentiles de la base con media lejos de 0,5 en algún grupo: " + ", ".join(lejos["columna"]))

# Debutante con ventana de 12 meses (enmienda): sin procesos del proveedor en los 365 días anteriores.
# Sale de la misma columna de historial que usan cuota y recurrencia (verificada en la celda 0).
if "ind_procesos_proveedor_12m" not in df.columns:
    raise SystemExit("No encuentro ind_procesos_proveedor_12m: corre primero la celda 0 y pégame su salida.")
n12 = a_float(df["ind_procesos_proveedor_12m"])
df["ind_proveedor_debutante_365"] = np.where(n12.isna(), np.nan, (n12 == 0).astype(float))

# 2015 queda solo como historial (enmienda): se retira de lo que se puntúa
DF_TODO = df.copy()     # base completa (con 2015): de aquí salen las distribuciones del recálculo
quitados = df.loc[df["anio_proc"] < ANIO_INICIO].groupby("grupo").size()
df = df.loc[df["anio_proc"] >= ANIO_INICIO].reset_index(drop=True)
say(f"\nProcesos anteriores a {ANIO_INICIO} retirados (siguen sirviendo de historial): {int(quitados.sum()):,}")
say(f"Procesos puntuables: {len(df):,}")

TAM_REAL = df.groupby("grupo").size()            # tamaño de cada grupo puntuable en la base completa
if MODO == "RAPIDO":
    df = (df.sample(frac=1.0, random_state=SEMILLA)
            .groupby("grupo", sort=False).head(RAPIDO_MAX_GRUPO)
            .sort_values("ocid").reset_index(drop=True))
    say(f"\nModo RAPIDO: se trabaja con {len(df):,} procesos (hasta {RAPIDO_MAX_GRUPO:,} por grupo).")

tab = (df.groupby("grupo").agg(procesos=("ocid", "size"),
                               de_2023=("anio_proc", lambda s: int((s == 2023).sum())),
                               anio_min=("anio_proc", "min"), anio_max=("anio_proc", "max"),
                               millones=("monto_final", lambda s: round(s.sum() / 1e6, 1))))
tab.insert(0, "en_la_base", TAM_REAL)
tab.insert(1, f"quitados_antes_{ANIO_INICIO}", quitados.reindex(tab.index).fillna(0).astype(int))
tab = tab.sort_values("en_la_base", ascending=False)
say("\nGrupos comparables (familia | período):")
say(tab.to_string())
say("\nRecordatorio: planA_validacion_sri NO se cargó. Ninguna cifra de esta corrida usa la marca del SRI.")
say(f"[celda 2: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CUADERNO 23 (EXPLICACION) · CELDA E3. Codificación de las variables (igual a la celda 3 del cuaderno 17)
#   rango      -> percentil del grupo (ya viene en la base); si falta el dato queda vacío (NaN)
#   rango_pos  -> percentil entre los procesos donde el fenómeno ocurre; 0 cuando no ocurre
#                 (cero real: no hubo historial o no hubo recompra), nunca un relleno
#   binaria    -> 0/1; vacío si falta
#   dif        -> diferencia con la mediana de oferentes del grupo, llevada a 0-1 dentro del grupo;
#                 1 cuando hubo un solo oferente (el extremo real de la escala)
# =====================================================================================
empezar("E3")
t_celda = time.time()

def pct_mitad_salto(s, grupos):
    """Percentil a mitad de salto dentro de cada grupo (los vacíos quedan vacíos)."""
    r = s.groupby(grupos).rank(method="average")
    n = s.notna().groupby(grupos).transform("sum")
    return (r - 0.5) / n

VARIABLES = {
    # nombre             columna en la base                  tipo         entra a la línea base
    "precio_alto":       ("pct_precio_alto",                 "rango",     True),
    "monto_capital":     ("pct_monto_capital",               "rango",     True),
    "proveedor_nuevo":   ("pct_proveedor_nuevo",             "rango",     True),
    "plazo_corto":       ("pct_plazo_corto",                 "rango",     True),
    "firma_rapida":      ("pct_firma_rapida",                "rango",     True),
    "recompra_30d":      ("pct_fraccionamiento_pos",         "rango_pos", True),
    "cuota_entidad":     ("pct_cuota_pos",                   "rango_pos", True),
    "recurrencia_par":   ("pct_recurrencia_pos",             "rango_pos", True),
    "oferente_unico":    ("ind_oferente_unico",              "binaria",   True),
    "poca_concurrencia": ("dif_concurrencia_grupo",          "dif",       True),
    # primera_vez_ent sale de los modelos: con la ventana de 12 meses coincide exactamente con
    # "recurrencia = 0", así que no aporta información nueva (enmienda del 16-oct-2026)
    "debutante":         ("ind_proveedor_debutante_365",     "binaria",   True),
}
# columnas de origen de las variables en dos partes: la que dice si el fenómeno ocurre (> 0) y la
# que mide su tamaño (solo se usa si hubiera que reconstruir el percentil)
ORIGEN_POS = {"recompra_30d":    ["ind_fraccionamiento_marca", "ind_fraccionamiento_monto_30d"],
              "cuota_entidad":   ["ind_cuota_prov_en_entidad_12m"],
              "recurrencia_par": ["ind_recurrencia_par_12m"]}
MAGNITUD_POS = {"recompra_30d":    ["ind_fraccionamiento_monto_30d"],
                "cuota_entidad":   ["ind_cuota_prov_en_entidad_12m"],
                "recurrencia_par": ["ind_recurrencia_par_12m"]}

def primera(cols):
    return next((c for c in cols if c in df.columns), None)

# Si una columna no existe con el nombre esperado, se reconstruye desde su origen (y se avisa)
RECONSTRUIDAS = []
for v, origenes in MAGNITUD_POS.items():
    col, origen = VARIABLES[v][0], primera(origenes)
    if col not in df.columns and origen is not None:
        mag = a_float(df[origen])
        df[col] = pct_mitad_salto(mag.where(mag > 0), df["grupo"])
        RECONSTRUIDAS.append(f"{v} desde {origen}")
col_nof = primera(["ind_n_oferentes", "n_oferentes"])
if col_nof is not None:
    nof = a_float(df[col_nof])
    if "ind_oferente_unico" not in df.columns:
        df["ind_oferente_unico"] = np.where(nof.isna(), np.nan, (nof == 1).astype(float))
        RECONSTRUIDAS.append(f"oferente_unico desde {col_nof}")
    if "dif_concurrencia_grupo" not in df.columns:
        df["dif_concurrencia_grupo"] = (nof.groupby(df["grupo"]).transform("median") - nof).where(nof >= 2)
        RECONSTRUIDAS.append(f"poca_concurrencia desde {col_nof}")

X = pd.DataFrame(index=df.index)
say("Variables del modelo y columna de origen:")
for v, (col, tipo, _) in list(VARIABLES.items()):
    if col not in df.columns:
        say(f"  {v:<18} NO ENCONTRADA ({col}) -> queda fuera")
        ALERTAS.append(f"Variable {v} no encontrada en la base (columna {col}).")
        VARIABLES.pop(v)
        continue
    s = a_float(df[col])
    nota = ""
    if tipo == "rango_pos":
        origen = primera(ORIGEN_POS[v])
        if origen is not None:
            o = a_float(df[origen])
            ocurre, no_ocurre = o > 0, o <= 0
            mal = int((s.isna() & ocurre).sum() + ((s > 0) & no_ocurre).sum())
            nota = f"control de ceros con {origen}: {mal:,} filas incoherentes" + ("  <- REVISAR" if mal else " (ok)")
            if mal:
                ALERTAS.append(f"{v}: {mal:,} filas donde el percentil y su origen no coinciden.")
            s = s.where(~(no_ocurre & s.isna()), 0.0)
        else:
            nota = "sin columna de origen: vacío se toma como 'no ocurre'"
            s = s.fillna(0.0)
    X[v] = s
    say(f"  {v:<18} {col:<32} {tipo:<10} cobertura {100 * s.notna().mean():5.1f} %   {nota}")
if RECONSTRUIDAS:
    say("Reconstruidas porque la columna esperada no existe:", "; ".join(RECONSTRUIDAS))
    ALERTAS.append("Variables reconstruidas desde su origen: " + "; ".join(RECONSTRUIDAS))
if X.shape[1] < 6:
    raise SystemExit("Faltan demasiadas variables. Pega en el chat la lista de columnas de la celda 2.")

# poca_concurrencia: escala 0-1 dentro de cada grupo, 1 = un solo oferente
if "poca_concurrencia" in X.columns:
    pc = X["poca_concurrencia"].copy()
    for g, idx in df.groupby("grupo").groups.items():
        vals = pc.loc[idx].dropna()
        if len(vals) >= 30:
            lo, hi = np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)
            if hi > lo:
                pc.loc[idx] = ((pc.loc[idx].clip(lo, hi) - lo) / (hi - lo))
            else:
                pc.loc[idx] = np.where(pc.loc[idx].notna(), 0.5, np.nan)
        else:
            pc.loc[idx] = np.where(pc.loc[idx].notna(), 0.5, np.nan)
    if "oferente_unico" in X.columns:
        pc = pc.where(~(X["oferente_unico"] == 1), 1.0)
        pc = pc.where(~X["oferente_unico"].isna(), np.nan)
    X["poca_concurrencia"] = pc

ORIENTADAS = [v for v, (_, _, base) in VARIABLES.items() if base]
TIPO = {v: t for v, (_, t, _) in VARIABLES.items()}
CORTO = {"precio_alto": "precio", "monto_capital": "capital", "proveedor_nuevo": "prov_nuevo",
         "plazo_corto": "plazo", "firma_rapida": "firma", "recompra_30d": "recompra",
         "cuota_entidad": "cuota", "recurrencia_par": "recurr", "oferente_unico": "of_unico",
         "poca_concurrencia": "poca_conc", "primera_vez_ent": "primera_vez", "debutante": "debutante"}

# control de orientación (alto = anómalo); la calibración ya se revisó en la celda 2
def control_orientacion(var, origen, signo):
    if var in X.columns and origen in df.columns:
        a, b = X[var], a_float(df[origen])
        ok = a.notna() & b.notna() & ((a > 0) if TIPO[var] == "rango_pos" else True)
        if ok.sum() > 100:
            r = spearmanr(a[ok], b[ok]).correlation
            estado = "ok" if np.sign(r) == signo else "REVISAR ORIENTACIÓN"
            if estado != "ok":
                ALERTAS.append(f"Orientación dudosa: {var} frente a {origen} (rho {r:+.2f}).")
            say(f"  {var:<18} vs {origen:<32} rho = {r:+.3f}  (esperado {'+' if signo > 0 else '-'})  {estado}")
say("\nControl de orientación (alto = más anómalo):")
control_orientacion("precio_alto", "ind_desv_precio_unit_z", +1)
control_orientacion("monto_capital", "ind_monto_sobre_capital", +1)
control_orientacion("proveedor_nuevo", "ind_antiguedad_dias", -1)
control_orientacion("plazo_corto", "ind_plazo_ofertas_dias", -1)
control_orientacion("firma_rapida", "ind_plazo_adj_firma_dias", -1)
control_orientacion("recompra_30d", "ind_fraccionamiento_monto_30d", +1)
control_orientacion("cuota_entidad", "ind_cuota_prov_en_entidad_12m", +1)
control_orientacion("recurrencia_par", "ind_recurrencia_par_12m", +1)
if col_nof is not None:
    control_orientacion("poca_concurrencia", col_nof, -1)

cob = (X.notna().groupby(df["grupo"]).mean() * 100).round(1)
cob.insert(0, "procesos", df.groupby("grupo").size())
say("\nCobertura de cada variable por grupo (% de procesos con dato):")
say(cob.sort_values("procesos", ascending=False).rename(columns=CORTO).to_string())

# (el cuaderno 17 guardaba aquí X y las llaves; el motor no las guarda)
say(f"[celda 3: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CUADERNO 23 (EXPLICACION) · CELDA E4. Patrones de disponibilidad (igual a la celda 4 del cuaderno 17)
#   - Se retiran del grupo las variables casi ausentes (<1 %) o constantes (>= 99,5 % un valor).
#   - Cada patrón con >= MIN_PATRON procesos tiene modelo propio ("grande").
#   - Un patrón menor se puntúa con el modelo grande que usa solo variables que él sí tiene;
#     si no hay ninguno, recibe un modelo propio ajustado sin la variable que le falta ("pequeño").
# =====================================================================================
empezar("E4")
def variables_del_grupo(Xg):
    usar, fuera = [], []
    for v in Xg.columns:
        s = Xg[v].dropna()
        if len(s) < max(30, 0.01 * len(Xg)):
            fuera.append(f"{v} (sin dato)")
        elif s.value_counts(normalize=True).iloc[0] >= 0.995:
            fuera.append(f"{v} (constante)")
        else:
            usar.append(v)
    return usar, fuera

def plan_de_modelos(Xg, usar):
    disp = Xg[usar].notna().to_numpy()
    codigo = (disp * (1 << np.arange(len(usar)))).sum(axis=1)
    texto = {c: "".join("1" if (c >> j) & 1 else "0" for j in range(len(usar))) for c in np.unique(codigo)}
    clave = pd.Series([texto[c] for c in codigo], index=Xg.index)
    conteo = clave.value_counts()
    grandes = [k for k, n in conteo.items() if n >= MIN_PATRON] or [conteo.index[0]]
    def vars_de(k): return [u for u, b in zip(usar, k) if b == "1"]
    asign = {}
    for k in conteo.index:
        if k in grandes:
            asign[k] = k
            continue
        mias = set(vars_de(k))
        cand = [g for g in grandes if set(vars_de(g)) <= mias]
        asign[k] = max(cand, key=lambda g: (len(vars_de(g)), conteo[g])) if cand else k
    modelos = sorted(set(asign.values()), key=lambda k: -conteo.get(k, 0))
    return clave, conteo, asign, {m: vars_de(m) for m in modelos}, set(grandes)

PLAN, NO_MODELADOS, filas = {}, {}, []
for g in df.groupby("grupo").size().sort_values(ascending=False).index:
    idx = df.index[df["grupo"] == g]
    Xg = X.loc[idx]
    usar, fuera = variables_del_grupo(Xg)
    if len(idx) < MIN_GRUPO or len(usar) < 2:
        NO_MODELADOS[g] = f"{len(idx)} procesos, {len(usar)} variables útiles"
        continue
    clave, conteo, asign, vars_modelo, grandes = plan_de_modelos(Xg, usar)
    PLAN[g] = dict(idx=idx, usar=usar, clave=clave, asign=asign, vars_modelo=vars_modelo, grandes=grandes)
    modelo_de = clave.map(asign)
    filas.append(dict(grupo=g, procesos=len(idx), variables=len(usar), patrones=len(conteo),
                      modelos_grandes=sum(m in grandes for m in vars_modelo),
                      modelos_pequenos=sum(m not in grandes for m in vars_modelo),
                      pct_en_modelo_principal=round(100 * (modelo_de == conteo.index[0]).mean(), 1),
                      pct_en_modelos_pequenos=round(100 * (~modelo_de.isin(grandes)).mean(), 2),
                      vars_modelo_principal=conteo.index[0].count("1"),
                      fuera=", ".join(fuera)))
say(pd.DataFrame(filas).to_string(index=False))
if NO_MODELADOS:
    say("\nGrupos que no se modelan:", NO_MODELADOS)

In [ ]:
# =====================================================================================
# CUADERNO 23 (EXPLICACION) · CELDA E5. Funciones de los modelos (iguales a la celda 5 del cuaderno 17)
# =====================================================================================
empezar("E5")
def pct_vs_ref(ref, x):
    """Percentil a mitad de salto de cada valor de x frente a la distribución de referencia ref."""
    ref = np.sort(np.asarray(ref, dtype=float))
    lo = np.searchsorted(ref, x, side="left")
    hi = np.searchsorted(ref, x, side="right")
    return (lo + 0.5 * (hi - lo)) / len(ref)

def ajustar_un_modelo(Xtr, Xsc, semilla, ref_max, ae_max):
    """Entrena los tres detectores con Xtr (procesos que tienen todas las variables del modelo)
    y puntúa Xsc. Devuelve percentiles (alto = más anómalo) frente a una referencia tomada de Xtr."""
    rng = np.random.default_rng(semilla)
    n, d = Xtr.shape
    R = Xtr[rng.choice(n, size=min(n, ref_max), replace=False)]
    out, tiempos = {}, {}
    # Isolation Forest
    t = time.time()
    iso = IsolationForest(n_estimators=IF_ARBOLES, max_samples=min(IF_SUBMUESTRA, n),
                          random_state=semilla, n_jobs=-1).fit(Xtr)
    out["if"] = pct_vs_ref(-iso.score_samples(R), -iso.score_samples(Xsc))
    tiempos["if"] = time.time() - t
    # Local Outlier Factor (modo novedad). Se ajusta sobre las filas distintas de la referencia.
    # Un proceso idéntico a una fila de la referencia recibe el LOF de esa fila, calculado sin
    # contarse a sí misma; así todos los procesos se miden con la misma regla.
    t = time.time()
    U, inv = np.unique(R, axis=0, return_inverse=True)
    inv = np.asarray(inv).ravel()
    if len(U) > LOF_VECINOS + 5:
        lof = LocalOutlierFactor(n_neighbors=LOF_VECINOS, novelty=True, n_jobs=-1).fit(U)
        lof_u = -lof.negative_outlier_factor_
        _, inv_all = np.unique(np.vstack([U, Xsc]), axis=0, return_inverse=True)
        inv_all = np.asarray(inv_all).ravel()
        pos = np.full(inv_all.max() + 1, -1)
        pos[inv_all[:len(U)]] = np.arange(len(U))
        m = pos[inv_all[len(U):]]
        s = np.empty(len(Xsc))
        s[m >= 0] = lof_u[m[m >= 0]]
        if (m < 0).any():
            s[m < 0] = -lof.score_samples(Xsc[m < 0])
        out["lof"] = pct_vs_ref(lof_u[inv], s)
    else:
        out["lof"] = np.full(len(Xsc), np.nan)
    tiempos["lof"] = time.time() - t
    # Autoencoder: red simétrica con capa intermedia de la mitad de las variables
    t = time.time()
    if d >= 3:
        tr_i = rng.choice(n, size=min(n, ae_max), replace=False)
        ae = MLPRegressor(hidden_layer_sizes=(max(2, d // 2),), activation="relu", solver="adam",
                          alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=200,
                          early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
                          random_state=semilla)
        ae.fit(Xtr[tr_i], Xtr[tr_i])
        err_ref = ((ae.predict(R) - R) ** 2).mean(axis=1)
        err_sc = ((ae.predict(Xsc) - Xsc) ** 2).mean(axis=1)
        out["ae"] = pct_vs_ref(err_ref, err_sc)
    else:
        out["ae"] = np.full(len(Xsc), np.nan)
    tiempos["ae"] = time.time() - t
    return out, tiempos

def top_k(valores, frac, semilla):
    """Marca el frac superior (exactamente k procesos con puntaje; empates al azar con semilla fija)."""
    v = np.asarray(valores, dtype=float)
    fin = np.isfinite(v)
    marca = np.zeros(len(v), dtype=bool)
    if fin.sum() == 0:
        return marca
    k = max(1, int(math.ceil(frac * fin.sum())))
    desempate = np.random.default_rng(semilla).random(len(v))
    orden = np.lexsort((desempate, -np.where(fin, v, -np.inf)))
    marca[orden[:k]] = True
    return marca

def rareza(Xg_df):
    """Percentil a mitad de salto de cada variable dentro del conjunto (alto = valor más alto)."""
    return (Xg_df.rank(method="average") - 0.5) / Xg_df.notna().sum()

def entrada_modelo(Xg_df):
    """Lo que reciben los modelos. 'percentil': los valores tal cual. 'normal_sup': la rareza de cada
    valor dentro del grupo en escala normal, con la mitad baja (sin riesgo) llevada a 0."""
    if REPRESENTACION == "normal_sup":
        z = norm.ppf(rareza(Xg_df).clip(1e-9, 1 - 1e-9).to_numpy(dtype=float))
        return pd.DataFrame(np.maximum(z, 0.0), index=Xg_df.index, columns=Xg_df.columns).where(Xg_df.notna())
    return Xg_df

def puntuar_grupo(plan, Xg_df, semilla, verbose=False):
    """Ajusta todos los modelos de un grupo. Devuelve percentiles por proceso, tiempos y rareza."""
    res = pd.DataFrame(index=Xg_df.index)
    Xm = entrada_modelo(Xg_df[plan["usar"]])
    for m in ["if", "lof", "ae"]:
        res[m] = np.nan
    res["modelo"] = ""
    t_mod = {"if": 0.0, "lof": 0.0, "ae": 0.0}
    clave = plan["clave"].loc[Xg_df.index]
    modelo_de = clave.map(plan["asign"])
    for j, (mk, vs) in enumerate(plan["vars_modelo"].items(), start=1):
        tm = time.time()
        destino = modelo_de.index[modelo_de == mk]
        if len(destino) == 0 or len(vs) == 0:
            continue
        tr_mask = Xg_df[vs].notna().all(axis=1)
        if tr_mask.sum() < 30:
            continue
        tope = (REF_MAX, AE_MAX_ENTRENO) if mk in plan["grandes"] else (HUERFANO_MAX, HUERFANO_MAX)
        out, tt = ajustar_un_modelo(Xm.loc[tr_mask, vs].to_numpy(dtype=float),
                                    Xm.loc[destino, vs].to_numpy(dtype=float), semilla, *tope)
        for m in out:
            res.loc[destino, m] = out[m]
            t_mod[m] += tt[m]
        res.loc[destino, "modelo"] = mk
        if verbose:
            say(f"      modelo {j}/{len(plan['vars_modelo'])}: {len(vs)} variables, entrena con "
                  f"{int(tr_mask.sum()):,}, puntúa {len(destino):,}  ({time.time() - tm:.0f} s)")
    res["ensamble"] = res[["if", "lof", "ae"]].mean(axis=1, skipna=True)
    rz = rareza(Xg_df[plan["usar"]])
    orient = [v for v in ORIENTADAS if v in plan["usar"]]
    res["base"] = rz[orient].mean(axis=1, skipna=True) if orient else np.nan
    res["n_vars_disp"] = Xg_df[plan["usar"]].notna().sum(axis=1)
    return res, t_mod, rz

def explicar(rz, cols, n=3):
    """Para cada fila, las n variables con valor más alto dentro del grupo."""
    a = rz[cols].to_numpy(dtype=float)
    a = np.where(np.isfinite(a), a, -1.0)
    orden = np.argsort(-a, axis=1)[:, :n]
    return [" | ".join(f"{cols[j]} {a[i, j]:.2f}" for j in orden[i] if a[i, j] >= 0) for i in range(len(a))]

def marcar_tops(res, semilla):
    for m in ["ensamble", "if", "lof", "ae", "base"]:
        res[f"top_{m}"] = top_k(res[m], CORTE_TOP, semilla)
    res["consenso"] = res[["top_if", "top_lof", "top_ae"]].sum(axis=1) >= 2
    return res
say("Funciones de los modelos cargadas.")

In [ ]:
# =====================================================================================
# CUADERNO 23 (EXPLICACION) · CELDA E6. Los mismos modelos de la celda E5, pero guardando lo ajustado
#   `ajustar_un_modelo_guardando` repite línea por línea `ajustar_un_modelo` (celda E5) y además
#   devuelve el Isolation Forest, el autoencoder y la muestra de referencia R de cada modelo.
#   `puntuar_grupo_guardando` repite `puntuar_grupo` y deja esos objetos en MODELOS[(grupo, patrón)].
#   La celda E7 comprueba que el resultado es el de la corrida congelada.
# =====================================================================================
empezar("E6")
MODELOS = {}

def ajustar_un_modelo_guardando(Xtr, Xsc, semilla, ref_max, ae_max):
    rng = np.random.default_rng(semilla)
    n, d = Xtr.shape
    R = Xtr[rng.choice(n, size=min(n, ref_max), replace=False)]
    out, tiempos = {}, {}
    # Isolation Forest
    t = time.time()
    iso = IsolationForest(n_estimators=IF_ARBOLES, max_samples=min(IF_SUBMUESTRA, n),
                          random_state=semilla, n_jobs=-1).fit(Xtr)
    out["if"] = pct_vs_ref(-iso.score_samples(R), -iso.score_samples(Xsc))
    tiempos["if"] = time.time() - t
    # Local Outlier Factor (modo novedad), igual que en la celda E5
    t = time.time()
    U, inv = np.unique(R, axis=0, return_inverse=True)
    inv = np.asarray(inv).ravel()
    if len(U) > LOF_VECINOS + 5:
        lof = LocalOutlierFactor(n_neighbors=LOF_VECINOS, novelty=True, n_jobs=-1).fit(U)
        lof_u = -lof.negative_outlier_factor_
        _, inv_all = np.unique(np.vstack([U, Xsc]), axis=0, return_inverse=True)
        inv_all = np.asarray(inv_all).ravel()
        pos = np.full(inv_all.max() + 1, -1)
        pos[inv_all[:len(U)]] = np.arange(len(U))
        m = pos[inv_all[len(U):]]
        s = np.empty(len(Xsc))
        s[m >= 0] = lof_u[m[m >= 0]]
        if (m < 0).any():
            s[m < 0] = -lof.score_samples(Xsc[m < 0])
        out["lof"] = pct_vs_ref(lof_u[inv], s)
    else:
        out["lof"] = np.full(len(Xsc), np.nan)
    tiempos["lof"] = time.time() - t
    # Autoencoder
    t = time.time()
    ae = None
    if d >= 3:
        tr_i = rng.choice(n, size=min(n, ae_max), replace=False)
        ae = MLPRegressor(hidden_layer_sizes=(max(2, d // 2),), activation="relu", solver="adam",
                          alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=200,
                          early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
                          random_state=semilla)
        ae.fit(Xtr[tr_i], Xtr[tr_i])
        err_ref = ((ae.predict(R) - R) ** 2).mean(axis=1)
        err_sc = ((ae.predict(Xsc) - Xsc) ** 2).mean(axis=1)
        out["ae"] = pct_vs_ref(err_ref, err_sc)
    else:
        out["ae"] = np.full(len(Xsc), np.nan)
    tiempos["ae"] = time.time() - t
    return out, tiempos, dict(iso=iso, ae=ae, R=R)

def puntuar_grupo_guardando(plan, Xg_df, semilla, grupo, verbose=False):
    res = pd.DataFrame(index=Xg_df.index)
    Xm = entrada_modelo(Xg_df[plan["usar"]])
    for m in ["if", "lof", "ae"]:
        res[m] = np.nan
    res["modelo"] = ""
    t_mod = {"if": 0.0, "lof": 0.0, "ae": 0.0}
    clave = plan["clave"].loc[Xg_df.index]
    modelo_de = clave.map(plan["asign"])
    for j, (mk, vs) in enumerate(plan["vars_modelo"].items(), start=1):
        tm = time.time()
        destino = modelo_de.index[modelo_de == mk]
        if len(destino) == 0 or len(vs) == 0:
            continue
        tr_mask = Xg_df[vs].notna().all(axis=1)
        if tr_mask.sum() < 30:
            continue
        tope = (REF_MAX, AE_MAX_ENTRENO) if mk in plan["grandes"] else (HUERFANO_MAX, HUERFANO_MAX)
        out, tt, objetos = ajustar_un_modelo_guardando(Xm.loc[tr_mask, vs].to_numpy(dtype=float),
                                                       Xm.loc[destino, vs].to_numpy(dtype=float), semilla, *tope)
        for m in out:
            res.loc[destino, m] = out[m]
            t_mod[m] += tt[m]
        res.loc[destino, "modelo"] = mk
        MODELOS[(grupo, mk)] = dict(vars=list(vs), **objetos)
        if verbose:
            say(f"      modelo {j}/{len(plan['vars_modelo'])}: {len(vs)} variables, entrena con "
                f"{int(tr_mask.sum()):,}, puntúa {len(destino):,}  ({time.time() - tm:.0f} s)")
    res["ensamble"] = res[["if", "lof", "ae"]].mean(axis=1, skipna=True)
    rz = rareza(Xg_df[plan["usar"]])
    orient = [v for v in ORIENTADAS if v in plan["usar"]]
    res["base"] = rz[orient].mean(axis=1, skipna=True) if orient else np.nan
    res["n_vars_disp"] = Xg_df[plan["usar"]].notna().sum(axis=1)
    return res, t_mod, rz
say("Funciones definidas. No se ajustó ningún modelo todavía.")

In [ ]:
# =====================================================================================
# CUADERNO 23 (EXPLICACION) · CELDA E7. Reajuste con la semilla congelada y control de reproducción
#   Ajusta los modelos de los 13 grupos con SEMILLA (la de la corrida congelada), los guarda y compara
#   por `ocid` con puntajes.parquet: puntajes de Isolation Forest, LOF, autoencoder, ensamble e índice
#   simple, lista del 1 % superior y texto de `explica`. Si algo no coincide, se detiene: no se explica
#   con modelos que no son los de la lista. Solo imprime resultados por grupo, nunca procesos.
# =====================================================================================
empezar("E7")
t0 = time.time()
partes_n, explica_n = [], {}
for g, plan in PLAN.items():
    tg = time.time()
    res_g, t_mod, rz = puntuar_grupo_guardando(plan, X.loc[plan["idx"]], SEMILLA, g,
                                               verbose=len(plan["idx"]) >= 50_000)
    res_g = marcar_tops(res_g, SEMILLA)
    orient = [v for v in ORIENTADAS if v in plan["usar"]]
    sel = res_g.index[res_g["top_ensamble"]]
    ex = pd.Series("", index=res_g.index)
    ex.loc[sel] = explicar(rz.loc[sel], orient)
    res_g["explica"] = ex
    res_g["grupo"] = g
    partes_n.append(res_g)
    say(f"  {g:<42} {len(res_g):>8,} procesos  {time.time() - tg:7.1f} s")
RES_N = pd.concat(partes_n).sort_index()
RES_N["ocid"] = df.loc[RES_N.index, "ocid"].to_numpy()
say(f"\nTiempo de ajuste: {(time.time() - t0) / 60:.1f} min | modelos guardados: {len(MODELOS)}")

# --- lo congelado (se lee solo lo necesario; nada se imprime por proceso) ---
ruta_c = CONGELADA.replace("\\", "/")
cong = duckdb.connect().execute(
    f'SELECT ocid, grupo, "if" AS p_if, lof AS p_lof, ae AS p_ae, ensamble, base, top_ensamble, priorizado, explica '
    f"FROM read_parquet('{ruta_c}')").df().set_index("ocid")
cong["explica"] = cong["explica"].fillna("")
cong["priorizado"] = cong["priorizado"].fillna(False).astype(bool)
cong["top_ensamble"] = cong["top_ensamble"].fillna(False).astype(bool)
if cong.index.duplicated().any() or RES_N["ocid"].duplicated().any():
    raise SystemExit("Hay ocid repetidos en lo congelado o en lo reajustado: para y avisa.")
solo_c = len(cong.index.difference(RES_N["ocid"]))
solo_n = len(pd.Index(RES_N["ocid"]).difference(cong.index))
say(f"Procesos congelados: {len(cong):,} | reajustados: {len(RES_N):,} | solo en lo congelado {solo_c} | solo en lo reajustado {solo_n}")
if solo_c or solo_n:
    raise SystemExit("Los procesos puntuados no son los mismos que los de la corrida congelada: para y avisa.")

N_ = RES_N.set_index("ocid")
C_ = cong.reindex(N_.index)
filas = []
for g in PLAN:
    m = (N_["grupo"] == g).to_numpy()
    fila = dict(grupo=g, procesos=int(m.sum()))
    ok_nan = True
    for nom_n, nom_c in [("if", "p_if"), ("lof", "p_lof"), ("ae", "p_ae"), ("ensamble", "ensamble"), ("base", "base")]:
        a, b = N_.loc[m, nom_n].to_numpy(float), C_.loc[m, nom_c].to_numpy(float)
        if not np.array_equal(np.isnan(a), np.isnan(b)):
            ok_nan = False
        fin = np.isfinite(a) & np.isfinite(b)
        fila[f"dif_{nom_n}"] = float(np.max(np.abs(a[fin] - b[fin]))) if fin.any() else 0.0
    ta, tb = N_.loc[m, "top_ensamble"].to_numpy(bool), C_.loc[m, "top_ensamble"].to_numpy(bool)
    union = int((ta | tb).sum())
    fila["jaccard_lista"] = (int((ta & tb).sum()) / union) if union else 1.0
    mm = m & C_["top_ensamble"].to_numpy(bool)
    ea, eb = N_.loc[mm, "explica"].to_numpy(str), C_.loc[mm, "explica"].to_numpy(str)
    fila["explica_igual_pct"] = 100.0 * float((ea == eb).mean()) if len(ea) else 100.0
    fila["nan_igual"] = ok_nan
    filas.append(fila)
REPRO = pd.DataFrame(filas)
say("\nControl de reproducción por grupo (diferencia máxima de puntaje, Jaccard del 1 % superior y texto de `explica`):")
say(REPRO.assign(**{c: REPRO[c].map(lambda v: f"{v:.1e}") for c in REPRO.columns if c.startswith("dif_")}).to_string(index=False))
dif_max = float(REPRO[[c for c in REPRO.columns if c.startswith("dif_")]].to_numpy().max())
reproduce = (dif_max <= TOL_REPRODUCCION and bool(REPRO["nan_igual"].all())
             and bool((REPRO["jaccard_lista"] == 1.0).all()) and bool((REPRO["explica_igual_pct"] == 100.0).all()))
say(f"\nDiferencia máxima de cualquier puntaje: {dif_max:.2e} (tolerancia {TOL_REPRODUCCION:.0e}) | "
    f"Jaccard mínimo de la lista: {REPRO['jaccard_lista'].min():.3f} | `explica` igual: {REPRO['explica_igual_pct'].min():.1f} %")
REPRO.to_csv(os.path.join(DIR_RUN, "X4_reproduccion_por_grupo.csv"), index=False)
if not reproduce:
    raise SystemExit("Los modelos reajustados NO reproducen los puntajes congelados: no se explica nada. Avisa y pega esta salida.")
say("Reproducción exacta: los modelos guardados son los de la lista congelada.")

# --- la lista del objetivo 6 ---
LISTA_OCID = cong.index[cong["priorizado"]]
lista_csv = pd.read_csv(LISTA_CSV, usecols=["ocid"])
if len(LISTA_OCID) != N_LISTA_ESPERADO or set(lista_csv["ocid"]) != set(LISTA_OCID):
    raise SystemExit("`priorizado` no coincide con lista_objetivo6.csv: para y avisa.")
say(f"Procesos de la lista del objetivo 6: {len(LISTA_OCID):,} (coinciden con lista_objetivo6.csv)")

In [ ]:
# =====================================================================================
# CUADERNO 23 (EXPLICACION) · CELDA E8. Error por variable del autoencoder en los procesos de la lista
#   El puntaje del autoencoder de un proceso es el promedio, sobre las variables del modelo, del error
#   cuadrático de reconstrucción. Aquí se descompone ese promedio: la parte de cada variable
#   (error de la variable / suma de los errores) y el percentil de ese error frente al de la
#   muestra de referencia del modelo, en la misma variable. Es una descomposición exacta del puntaje,
#   no una estimación. Descriptivo (sección 11 del preregistro).
#   Versión 2 (23-nov-2026): la red se evalúa sobre el lote completo del modelo, como en la corrida
#   congelada, y se toma de ahí la fila de cada proceso de la lista. La versión 1 la evaluaba solo con
#   los procesos de la lista y el control se detuvo (diferencia de percentil 3,11e-04): el resultado
#   numérico de la red cambia en el último decimal según cuántas filas se le pasan a la vez, y con
#   empates en la muestra de referencia eso mueve el percentil. Esta versión imprime ese diagnóstico.
# =====================================================================================
empezar("E8")
t0 = time.time()
VARS_TODAS = list(X.columns)
IDX_LISTA = df.index[df["ocid"].isin(set(LISTA_OCID))]
if len(IDX_LISTA) != N_LISTA_ESPERADO:
    raise SystemExit("No se encuentran los 3.613 procesos de la lista en la base: para y avisa.")

# filas de la lista por grupo y modelo
MODELO_DE = {}
for g, plan in PLAN.items():
    mod = plan["clave"].map(plan["asign"])
    sel = IDX_LISTA.intersection(plan["idx"])
    for mk, sub in mod.loc[sel].groupby(mod.loc[sel]):
        MODELO_DE[(g, mk)] = sub.index
if sum(len(v) for v in MODELO_DE.values()) != N_LISTA_ESPERADO:
    raise SystemExit("Hay procesos de la lista fuera de los modelos ajustados: para y avisa.")

EXP = pd.DataFrame(index=IDX_LISTA)
EXP["ocid"] = df.loc[IDX_LISTA, "ocid"].to_numpy()
EXP["grupo"] = df.loc[IDX_LISTA, "grupo"].to_numpy()
EXP["modelo"] = ""
EXP["n_vars_modelo"] = 0
EXP["if_f_x"] = np.nan
for v in VARS_TODAS:
    EXP[f"ae_parte_{v}"] = np.nan
    EXP[f"ae_pct_{v}"] = np.nan
    EXP[f"phi_{v}"] = np.nan
    EXP[f"phi_ee_{v}"] = np.nan

XM_CACHE = {}
def entrada_del_grupo(g):
    """Entrada de los modelos (celda E5) de todos los procesos del grupo; se calcula una vez por grupo."""
    if g not in XM_CACHE:
        Xg_df = X.loc[PLAN[g]["idx"]]                      # mismo orden de operaciones que la celda E6
        XM_CACHE[g] = entrada_modelo(Xg_df[PLAN[g]["usar"]])
    return XM_CACHE[g]

def bloque_de_empate(refs_ordenadas, e):
    """Cuántas filas de la referencia tienen un error igual al de cada proceso, salvo ruido de redondeo."""
    return (np.searchsorted(refs_ordenadas, e * (1 + 1e-12), side="right")
            - np.searchsorted(refs_ordenadas, e * (1 - 1e-12), side="left"))

say("E8, versión 2: el error de la red se calcula sobre el lote completo del modelo, como en la corrida congelada.")
chequeo_ae, sin_ae, claves_ae, diag_lote = [], 0, [], []
n_fuera, n_dif = 0, 0
for (g, mk), filas_idx in MODELO_DE.items():
    plan, M = PLAN[g], MODELOS[(g, mk)]
    vs = M["vars"]
    EXP.loc[filas_idx, "modelo"] = mk
    EXP.loc[filas_idx, "n_vars_modelo"] = len(vs)
    if M["ae"] is None:
        sin_ae += len(filas_idx)
        continue
    Xm_g = entrada_del_grupo(g)
    # todos los procesos que puntúa este modelo (igual que en la celda E6)
    modelo_de = plan["clave"].loc[plan["idx"]].map(plan["asign"])
    destino = modelo_de.index[modelo_de == mk]
    pos = destino.get_indexer(filas_idx)
    if (pos < 0).any():
        raise SystemExit("Hay procesos de la lista que el modelo no puntúa: para y avisa.")
    Xfull = Xm_g.loc[destino, vs].to_numpy(dtype=float)
    R = M["R"]
    err_ref_v = (M["ae"].predict(R) - R) ** 2                 # (n_ref, d)
    err_full_v = (M["ae"].predict(Xfull) - Xfull) ** 2        # (n_destino, d)
    err_tot_full = err_full_v.mean(axis=1)
    err_v, err_tot = err_full_v[pos], err_tot_full[pos]       # las filas de la lista
    ref_tot = err_ref_v.mean(axis=1)
    # control: el percentil del puntaje total reproduce el congelado
    p = pct_vs_ref(ref_tot, err_tot)
    refs = np.sort(ref_tot)
    d_i = np.abs(p - RES_N.loc[filas_idx, "ae"].to_numpy(float))
    # un cambio en el último decimal solo puede mover un percentil dentro del bloque de filas de la referencia
    # con un error igual al del proceso salvo ruido de redondeo (1e-12 en términos relativos)
    bloque = bloque_de_empate(refs, err_tot) / len(refs)
    chequeo_ae.append(float(np.max(d_i)))
    n_fuera += int((d_i > TOL_REPRODUCCION + bloque).sum())
    n_dif += int((d_i > TOL_REPRODUCCION).sum())
    claves_ae.append((g, mk, len(R), len(destino), len(filas_idx)))
    # diagnóstico de la versión 1: el mismo error calculado solo con los procesos de la lista
    Xs_v1 = Xm_g.loc[filas_idx, vs].to_numpy(dtype=float)
    ev_v1 = (M["ae"].predict(Xs_v1) - Xs_v1) ** 2
    e_v1 = ev_v1.mean(axis=1)
    dp = np.abs(pct_vs_ref(ref_tot, e_v1) - p)
    iw = int(np.argmax(dp))
    empate = int(bloque_de_empate(refs, err_tot[iw:iw + 1])[0])
    dpv = max(float(np.max(np.abs(pct_vs_ref(np.sort(err_ref_v[:, j]), ev_v1[:, j])
                                  - pct_vs_ref(np.sort(err_ref_v[:, j]), err_v[:, j])))) for j in range(len(vs)))
    diag_lote.append((float(dp.max()), float(np.max(np.abs(e_v1 - err_tot) / np.maximum(err_tot, 1e-300))),
                      int((e_v1 != err_tot).sum()), empate, len(filas_idx),
                      float(np.max(np.abs(ev_v1 - err_v) / np.maximum(err_v, 1e-300))), dpv))
    # parte de cada variable y percentil de su error
    suma = err_v.sum(axis=1)
    parte = np.where(suma[:, None] > 0, err_v / np.where(suma[:, None] > 0, suma[:, None], 1.0), np.nan)
    for j, v in enumerate(vs):
        ref_j = np.sort(err_ref_v[:, j])
        EXP.loc[filas_idx, f"ae_parte_{v}"] = parte[:, j]
        EXP.loc[filas_idx, f"ae_pct_{v}"] = pct_vs_ref(ref_j, err_v[:, j])
say(f"Procesos de la lista: {len(EXP):,} en {len(MODELO_DE)} modelos | sin autoencoder (menos de 3 variables): {sin_ae}")
dif_ae = max(chequeo_ae) if chequeo_ae else 0.0
say(f"Control: diferencia máxima entre el percentil del error total recalculado y el puntaje `ae` congelado: {dif_ae:.2e}")
if diag_lote:
    dl = np.array(diag_lote, dtype=float)
    say(f"Diagnóstico (calcular el error solo con la lista, como en la versión 1): diferencia relativa máxima del error "
        f"{dl[:, 1].max():.1e} | procesos con el error distinto en algún decimal {int(dl[:, 2].sum())} de {int(dl[:, 4].sum())} | "
        f"diferencia máxima de percentil {dl[:, 0].max():.2e} | filas de la referencia con error casi igual en el caso más afectado {int(dl[np.argmax(dl[:, 0]), 3])}")
    say(f"  Por variable: diferencia relativa máxima del error {dl[:, 5].max():.1e} | diferencia máxima del percentil de una variable {dl[:, 6].max():.2e} "
        f"(con empates en la referencia, un cambio en el último decimal mueve el percentil de una variable en esa medida)")
if n_fuera == 0 and n_dif > 0:
    ALERTAS.append(f"E8: {n_dif} procesos de la lista con percentil del autoencoder distinto del congelado (máximo {dif_ae:.1e}), "
                   f"todos dentro del bloque de empates de la referencia")
    say(f"AVISO: {n_dif} procesos difieren del congelado, todos dentro del bloque de filas de la referencia con error casi igual: "
        f"es un empate, no un error del cálculo. Queda en el registro.")
if n_fuera > 0:
    k = int(np.argmax(chequeo_ae))
    g_, mk_, nref_, ndest_, nlis_ = claves_ae[k]
    say(f"Peor modelo: {g_} | variables {len(MODELOS[(g_, mk_)]['vars'])} | referencia {nref_:,} | procesos del modelo {ndest_:,} | "
        f"procesos de la lista {nlis_} | diferencia {chequeo_ae[k]:.2e} | procesos fuera de los empates: {n_fuera}")
    raise SystemExit("El error por variable no reconstruye el puntaje del autoencoder: para y avisa.")
try:
    _fuente = get_ipython().user_ns["In"][-1]
    huella_e8 = hashlib.sha256(_fuente.encode("utf-8")).hexdigest()[:16]
except Exception:
    huella_e8 = "no disponible"
with open(os.path.join(DIR_RUN, "E8_version2.json"), "w", encoding="utf-8") as f_:
    json.dump(dict(motivo="la version 1 de E8 se detuvo: percentil del error total recalculado con solo la lista difirio del congelado",
                   huella_codigo_E1=HUELLA_CODIGO, huella_celda_E8_v2=huella_e8,
                   control_dif_max=dif_ae, procesos_con_diferencia=n_dif, procesos_fuera_de_empates=n_fuera, diagnostico_version1=dict(
                       dif_percentil_max=float(dl[:, 0].max()) if diag_lote else 0.0,
                       dif_relativa_error_max=float(dl[:, 1].max()) if diag_lote else 0.0,
                       procesos_error_distinto=int(dl[:, 2].sum()) if diag_lote else 0,
                       empate_caso_mas_afectado=int(dl[np.argmax(dl[:, 0]), 3]) if diag_lote else 0,
                       dif_relativa_error_por_variable_max=float(dl[:, 5].max()) if diag_lote else 0.0,
                       dif_percentil_por_variable_max=float(dl[:, 6].max()) if diag_lote else 0.0),
                   fecha=datetime.now().isoformat(timespec="seconds")), f_, ensure_ascii=False, indent=1, default=float)
say(f"({(time.time() - t0):.0f} s)")

In [ ]:
# =====================================================================================
# CUADERNO 23 (EXPLICACION) · CELDA E9. Valores de Shapley de Isolation Forest en los procesos de la lista
#   Función explicada: f(x) = -score_samples(x), el puntaje de anomalía del bosque (más alto = más raro);
#   es la cantidad de la que sale el percentil de Isolation Forest. No es el percentil del ensamble.
#   Método: Shapley "intervencional" con fondo R (la muestra de referencia del propio modelo): para cada
#   proceso se sortean N_CADENAS/2 pares (fondo, permutación); la cadena pasa las variables del fondo a
#   las del proceso en el orden de la permutación y cada variable recibe lo que cambia f; cada permutación
#   se usa también al revés. Cada cadena cumple la eficiencia exacta (la suma de los aportes es f(x) menos
#   f(fondo)), y el promedio de las cadenas es una estimación insesgada del Shapley, con error estándar.
#   Antes de explicar la lista, el método se compara con el Shapley EXACTO (los 2^d subconjuntos) en unos
#   pocos procesos, con un fondo pequeño. Supone variables independientes en el fondo: es una limitación.
#   (La comprobación de eficiencia por cadena es una identidad numérica; lo que valida la asignación de
#   cada aporte a su variable es la comparación con el Shapley exacto.)
# =====================================================================================
empezar("E9")

def _aportes(iso, xs, B, pos, f_b, f_x):
    """Aporte de cada variable en una cadena (m procesos, h cadenas, d variables).
    pos[..., j] = lugar de la variable j en la permutación. Devuelve (aportes, F) con F = f a lo largo de la cadena."""
    m, h, d = pos.shape
    F = np.empty((m, h, d + 1))
    F[..., 0] = f_b
    F[..., d] = f_x[:, None]
    if d > 1:
        Z = np.empty((m, h, d - 1, d))
        for i in range(1, d):
            Z[:, :, i - 1, :] = np.where(pos < i, xs[:, None, :], B)
        F[..., 1:d] = -iso.score_samples(Z.reshape(-1, d)).reshape(m, h, d - 1)
    D = F[..., 1:] - F[..., :-1]                      # D[..., k]: aporte de la variable que está en el lugar k
    return np.take_along_axis(D, pos, axis=2), F

def shapley_if(iso, x_rows, fondo, n_cadenas, rng, bloque=40):
    """Shapley intervencional de f = -iso.score_samples con fondo `fondo`. Devuelve (phi, error estándar, f(x), control)."""
    n, d = x_rows.shape
    h = n_cadenas // 2
    phi, ee = np.zeros((n, d)), np.zeros((n, d))
    f_x_todo = -iso.score_samples(x_rows)
    peor_eficiencia = 0.0
    for a in range(0, n, bloque):
        xs = x_rows[a:a + bloque]
        m = len(xs)
        B = fondo[rng.integers(0, len(fondo), size=(m, h))]                    # (m, h, d)
        orden = np.argsort(rng.random((m, h, d)), axis=2)
        pos = np.argsort(orden, axis=2)
        f_b = -iso.score_samples(B.reshape(-1, d)).reshape(m, h)
        f_x = f_x_todo[a:a + bloque]
        ap1, _ = _aportes(iso, xs, B, pos, f_b, f_x)
        ap2, _ = _aportes(iso, xs, B, d - 1 - pos, f_b, f_x)                   # la misma permutación al revés
        for ap in (ap1, ap2):
            peor_eficiencia = max(peor_eficiencia, float(np.abs(ap.sum(axis=2) - (f_x[:, None] - f_b)).max()))
        par = 0.5 * (ap1 + ap2)                                                # (m, h, d)
        phi[a:a + bloque] = par.mean(axis=1)
        ee[a:a + bloque] = par.std(axis=1, ddof=1) / math.sqrt(h) if h > 1 else np.nan
    return phi, ee, f_x_todo, peor_eficiencia

def shapley_exacto(iso, x, fondo):
    """Shapley intervencional exacto de un proceso x con el fondo dado (enumera los 2^d subconjuntos)."""
    d = len(x)
    N = 1 << d
    masks = ((np.arange(N)[:, None] >> np.arange(d)) & 1).astype(bool)
    Z = np.where(masks[:, None, :], x[None, None, :], fondo[None, :, :])
    v = (-iso.score_samples(Z.reshape(-1, d)).reshape(N, len(fondo))).mean(axis=1)
    pop = masks.sum(axis=1)
    fact = [math.factorial(k) for k in range(d + 1)]
    phi = np.zeros(d)
    for j in range(d):
        sin = np.where(~masks[:, j])[0]
        con = sin | (1 << j)
        w = np.array([fact[k] * fact[d - k - 1] / fact[d] for k in pop[sin]])
        phi[j] = float((w * (v[con] - v[sin])).sum())
    return phi, float(v[N - 1] - v[0])

# --- control contra el Shapley exacto (modelo con más variables y más procesos de la lista) ---
t0 = time.time()
cand = sorted(MODELO_DE, key=lambda k: (-len(MODELOS[k]["vars"]), -len(MODELO_DE[k])))
k_ctrl = cand[0]
M = MODELOS[k_ctrl]
g_c, mk_c = k_ctrl
Xm_c = entrada_del_grupo(g_c)
filas_c = MODELO_DE[k_ctrl][:N_PROC_EXACTO]
x_c = Xm_c.loc[filas_c, M["vars"]].to_numpy(dtype=float)
fondo_c = M["R"][:N_FONDO_EXACTO]
rng_c = np.random.default_rng(SEMILLA_SHAP + 1)
phi_mc, ee_mc, f_x_c, efi_c = shapley_if(M["iso"], x_c, fondo_c, N_CADENAS_EXACTO, rng_c)
razones, dif_abs = [], []
for i in range(len(x_c)):
    ph, suma_ex = shapley_exacto(M["iso"], x_c[i], fondo_c)
    dif = np.abs(ph - phi_mc[i])
    dif_abs.append(float(dif.max()))
    for dj, ej in zip(dif, ee_mc[i]):
        razones.append(float(dj / ej) if ej > 1e-12 else (0.0 if dj <= 1e-9 else 1e9))
    if abs(ph.sum() - suma_ex) > 1e-9:
        raise SystemExit("El Shapley exacto no suma f(x) menos el promedio del fondo: para y avisa.")
say(f"Control contra el Shapley exacto: modelo de {len(M['vars'])} variables, {len(x_c)} procesos, fondo de {len(fondo_c)} filas, "
    f"{N_CADENAS_EXACTO} cadenas.")
say(f"  diferencia máxima {max(dif_abs):.2e} | mayor diferencia en errores estándar {max(razones):.2f} | "
    f"desvío máximo de la eficiencia por cadena {efi_c:.1e}")
if max(razones) > 5 or efi_c > 1e-9:
    raise SystemExit("El Shapley estimado no coincide con el exacto: para y avisa.")

# --- la lista ---
rng = np.random.default_rng(SEMILLA_SHAP)
efi_max = 0.0
for (g, mk), filas_idx in MODELO_DE.items():
    plan, M = PLAN[g], MODELOS[(g, mk)]
    vs = M["vars"]
    Xs = entrada_del_grupo(g).loc[filas_idx, vs].to_numpy(dtype=float)
    phi, ee, f_x, efi = shapley_if(M["iso"], Xs, M["R"], N_CADENAS, rng, bloque=BLOQUE_SHAP)
    efi_max = max(efi_max, efi)
    for j, v in enumerate(vs):
        EXP.loc[filas_idx, f"phi_{v}"] = phi[:, j]
        EXP.loc[filas_idx, f"phi_ee_{v}"] = ee[:, j]
    EXP.loc[filas_idx, "if_f_x"] = f_x
say(f"Shapley de la lista: {len(EXP):,} procesos, {N_CADENAS} cadenas cada uno, {len(MODELO_DE)} modelos | "
    f"desvío máximo de la eficiencia por cadena {efi_max:.1e} | {(time.time() - t0) / 60:.1f} min")
if efi_max > 1e-9:
    raise SystemExit("La eficiencia del Shapley no se cumple: para y avisa.")

In [ ]:
# =====================================================================================
# CUADERNO 23 (EXPLICACION) · CELDA E10. Tablas agregadas, archivos y registro
#   Junta las tres explicaciones de cada proceso de la lista: las tres variables de mayor rareza
#   (`explica`, de la corrida congelada), las de mayor parte del error del autoencoder y las de mayor
#   Shapley de Isolation Forest. Imprime solo conteos y porcentajes; el archivo por proceso queda en la
#   carpeta de la corrida (lleva `ocid`, ningún RUC) y no se pega en el chat.
# =====================================================================================
empezar("E10")
V = VARS_TODAS
n_lista = len(EXP)

def nombres_top(mat, k=3, solo_positivos=False):
    m = np.where(np.isfinite(mat), mat, -np.inf)
    if solo_positivos:
        m = np.where(m > 0, m, -np.inf)
    orden = np.argsort(-m, axis=1, kind="stable")[:, :k]
    sal = []
    for j in range(k):
        col = orden[:, j]
        ok = m[np.arange(len(m)), col] > -np.inf
        sal.append(np.where(ok, np.array(V, dtype=object)[col], ""))
    return sal, m

ae_mat = EXP[[f"ae_parte_{v}" for v in V]].to_numpy(float)
phi_mat = EXP[[f"phi_{v}" for v in V]].to_numpy(float)
ee_mat = EXP[[f"phi_ee_{v}" for v in V]].to_numpy(float)
(ae1, ae2, ae3), _ = nombres_top(ae_mat)
(if1, if2, if3), phi_m = nombres_top(phi_mat, solo_positivos=True)
for k, (a, b) in enumerate(zip((ae1, ae2, ae3), (if1, if2, if3)), start=1):
    EXP[f"ae_{k}"], EXP[f"if_{k}"] = a, b

def vars_explica(s):
    if not isinstance(s, str) or not s:
        return []
    return [p.rsplit(" ", 1)[0] for p in s.split(" | ")]
expl = [vars_explica(s) for s in cong.loc[EXP["ocid"], "explica"].to_numpy()]
for k in range(3):
    EXP[f"explica_{k + 1}"] = [e[k] if len(e) > k else "" for e in expl]

# separación del Shapley: ¿la variable principal se distingue de la segunda? (aprox., ignora la covarianza)
orden_phi = np.argsort(-phi_m, axis=1, kind="stable")
f_ = np.arange(n_lista)
a1, a2 = orden_phi[:, 0], orden_phi[:, 1]
ok2 = np.isfinite(phi_m[f_, a1]) & np.isfinite(phi_m[f_, a2]) & (phi_m[f_, a1] > -np.inf) & (phi_m[f_, a2] > -np.inf)
dif12 = phi_m[f_, a1] - phi_m[f_, a2]
ee12 = np.sqrt(ee_mat[f_, a1] ** 2 + ee_mat[f_, a2] ** 2)
EXP["if_separado"] = np.where(EXP["if_1"] == "", False, np.where(ok2, dif12 > 2 * ee12, True))

# ---------- T1: por variable, sobre toda la lista ----------
def pct(mask):
    return round(100 * float(np.mean(mask)), 1)
filas = []
for v in V:
    filas.append(dict(variable=v,
                      en_modelo_pct=pct(EXP[f"phi_{v}"].notna() | EXP[f"ae_parte_{v}"].notna()),
                      explica_primera_pct=pct(EXP["explica_1"] == v),
                      explica_en_las_3_pct=pct((EXP[["explica_1", "explica_2", "explica_3"]] == v).any(axis=1)),
                      if_principal_pct=pct(EXP["if_1"] == v),
                      if_en_las_3_pct=pct((EXP[["if_1", "if_2", "if_3"]] == v).any(axis=1)),
                      ae_principal_pct=pct(EXP["ae_1"] == v),
                      ae_en_las_3_pct=pct((EXP[["ae_1", "ae_2", "ae_3"]] == v).any(axis=1)),
                      phi_medio=round(float(np.nanmean(EXP[f"phi_{v}"])) if EXP[f"phi_{v}"].notna().any() else np.nan, 4),
                      ae_parte_media=round(float(np.nanmean(EXP[f"ae_parte_{v}"])) if EXP[f"ae_parte_{v}"].notna().any() else np.nan, 3)))
T1 = pd.DataFrame(filas).sort_values("if_principal_pct", ascending=False)
say(f"E10-1. Variable principal según cada explicación, sobre los {n_lista:,} procesos de la lista (% de procesos):")
say(T1.to_string(index=False))

# ---------- T3: concordancia entre explicaciones ----------
tiene_if, tiene_ae = (EXP["if_1"] != "").to_numpy(), (EXP["ae_1"] != "").to_numpy()
ex_set = [set(e) for e in expl]
def en_explica(col):
    return np.array([(c != "") and (c in s) for c, s in zip(EXP[col], ex_set)])
if_en_ex, ae_en_ex = en_explica("if_1"), en_explica("ae_1")
if_igual_ae = (EXP["if_1"] == EXP["ae_1"]).to_numpy() & tiene_if & tiene_ae
ae_en_if3 = np.array([(a != "") and (a in {x, y, z}) for a, x, y, z in zip(EXP["ae_1"], EXP["if_1"], EXP["if_2"], EXP["if_3"])])
ex1_igual_if = (EXP["explica_1"] == EXP["if_1"]).to_numpy() & tiene_if
ex1_igual_ae = (EXP["explica_1"] == EXP["ae_1"]).to_numpy() & tiene_ae
T3 = pd.DataFrame([
    dict(medida="Procesos con variable principal de Isolation Forest (Shapley positivo)", n=int(tiene_if.sum()), pct=pct(tiene_if)),
    dict(medida="Procesos con variable principal del autoencoder", n=int(tiene_ae.sum()), pct=pct(tiene_ae)),
    dict(medida="La principal de Isolation Forest está entre las 3 de `explica`", n=int(if_en_ex.sum()), pct=pct(if_en_ex[tiene_if])),
    dict(medida="La principal del autoencoder está entre las 3 de `explica`", n=int(ae_en_ex.sum()), pct=pct(ae_en_ex[tiene_ae])),
    dict(medida="La principal de Isolation Forest es la primera de `explica`", n=int(ex1_igual_if.sum()), pct=pct(ex1_igual_if[tiene_if])),
    dict(medida="La principal del autoencoder es la primera de `explica`", n=int(ex1_igual_ae.sum()), pct=pct(ex1_igual_ae[tiene_ae])),
    dict(medida="Isolation Forest y autoencoder tienen la misma variable principal", n=int(if_igual_ae.sum()),
         pct=pct(if_igual_ae[tiene_if & tiene_ae])),
    dict(medida="La principal del autoencoder está entre las 3 de Isolation Forest", n=int(ae_en_if3.sum()), pct=pct(ae_en_if3[tiene_ae & tiene_if])),
    dict(medida="Shapley: la principal se separa de la segunda (más de 2 errores estándar, aprox.)",
         n=int(EXP["if_separado"][tiene_if].sum()), pct=pct(EXP["if_separado"].to_numpy()[tiene_if])),
])
say("\nE10-2. Concordancia entre las tres explicaciones (el porcentaje usa como base los procesos que tienen ambas):")
say(T3.to_string(index=False))

# ---------- T2: por grupo ----------
def mas_frecuente(serie):
    s = serie[serie != ""]
    if s.empty:
        return "", 0.0
    vc = s.value_counts()
    return vc.index[0], round(100 * vc.iloc[0] / len(serie), 1)
filas, largo = [], []
for g, sub in EXP.groupby("grupo"):
    idx = sub.index
    fila = dict(grupo=g, procesos=len(sub))
    for etq, col in [("explica", "explica_1"), ("if", "if_1"), ("ae", "ae_1")]:
        v, p = mas_frecuente(sub[col])
        fila[f"{etq}_mas_frecuente"], fila[f"{etq}_pct"] = v, p
        for vv, n in sub[col][sub[col] != ""].value_counts().items():
            largo.append(dict(grupo=g, fuente=etq, variable=vv, procesos=int(n), pct=round(100 * n / len(sub), 1)))
    pos = np.array([EXP.index.get_loc(i) for i in idx])
    fila["if_en_explica_pct"] = pct(if_en_ex[pos][tiene_if[pos]]) if tiene_if[pos].any() else np.nan
    fila["ae_en_explica_pct"] = pct(ae_en_ex[pos][tiene_ae[pos]]) if tiene_ae[pos].any() else np.nan
    filas.append(fila)
T2 = pd.DataFrame(filas).sort_values("procesos", ascending=False)
T2L = pd.DataFrame(largo)
say("\nE10-3. Por grupo: variable principal más frecuente según cada explicación (% de los procesos del grupo en la lista):")
say(T2.to_string(index=False))

# ---------- archivos ----------
if any("ruc" in c.lower() for c in EXP.columns):
    raise SystemExit("El archivo por proceso trae una columna de RUC: para y avisa.")
T1.to_csv(os.path.join(DIR_RUN, "X1_variable_principal_global.csv"), index=False)
T2.to_csv(os.path.join(DIR_RUN, "X2_variable_principal_por_grupo.csv"), index=False)
T2L.to_csv(os.path.join(DIR_RUN, "X2b_variable_principal_por_grupo_largo.csv"), index=False)
T3.to_csv(os.path.join(DIR_RUN, "X3_concordancia.csv"), index=False)
EXP.to_csv(os.path.join(DIR_RUN, "explicacion_por_proceso.csv"), index=False)     # LOCAL: no se pega en el chat

archivos = {f: sha_completa(os.path.join(DIR_RUN, f)) for f in sorted(os.listdir(DIR_RUN)) if f.endswith(".csv")}
registro = dict(analisis="explicacion de la lista del objetivo 6", estatus="descriptivo, seccion 11 del preregistro (desviacion declarada en la seccion 12)",
                preregistro=FIRMA["huella_documento"], huella_codigo=HUELLA_CODIGO,
                puntajes_congelados=HUELLA_CONGELADA, lista_objetivo6=HUELLA_LISTA,
                huella_indicadores=HUELLA_IX, huella_procesos=HUELLA_PM, semilla=SEMILLA,
                reproduccion=dict(dif_maxima=dif_max, tolerancia=TOL_REPRODUCCION,
                                  grupos=REPRO.to_dict(orient="records")),
                shapley=dict(metodo="intervencional, permutaciones con inversa, fondo = referencia del modelo",
                             cadenas=N_CADENAS, funcion="-score_samples del Isolation Forest",
                             control_exacto=dict(fondo=N_FONDO_EXACTO, procesos=N_PROC_EXACTO, cadenas=N_CADENAS_EXACTO,
                                                 dif_max=max(dif_abs), razon_max_en_ee=max(razones), eficiencia_max=efi_max)),
                autoencoder=dict(control_percentil_max=max(chequeo_ae) if chequeo_ae else 0.0, sin_autoencoder=sin_ae),
                procesos_lista=n_lista, modelos=len(MODELO_DE), versiones=VERSIONES, alertas=ALERTAS,
                archivos=archivos, fecha=datetime.now().isoformat(timespec="seconds"), carpeta=DIR_RUN,
                minutos=round((time.time() - T_INICIO) / 60, 1))
with open(os.path.join(DIR_RUN, "registro_explicacion.json"), "w", encoding="utf-8") as f:
    json.dump(registro, f, ensure_ascii=False, indent=1, default=float)
say(f"\nRegistro escrito en {DIR_RUN} ({registro['minutos']} min en total)")
say("Pega en el chat SOLO resumen_para_chat.txt (agregado; no trae RUC ni procesos). "
    "explicacion_por_proceso.csv se queda en tu computadora.")
guardar_txt()